# 04 - Agrupamiento de Densidad Variable: OPTICS, Gráficos de Alcanzabilidad y HDBSCAN Profundo

## Curso de Machine Learning: Modelos No Supervisados

En los cuadernos anteriores vimos que el principal límite estructural de **DBSCAN** es el uso de un único umbral de densidad global $\varepsilon$ (`eps`). En mercados financieros reales coexistirán regímenes con dispersiones intrínsecas radicalmente distintas:
- **Bonos Soberanos / Utilities (Baja Volatilidad)**: Se agrupan en una región hiper-densa de radio muy pequeño ($\varepsilon_1 \approx 0.35$).
- **Emisores Growth / Mercados Emergentes (Alta Volatilidad)**: Forman un régimen legítimo pero mucho más disperso ($\varepsilon_2 \approx 1.20$).

Si fijamos $\varepsilon = 0.35$, el régimen emergente es destruido y etiquetado como ruido (`-1`). Si fijamos $\varepsilon = 1.20$, el régimen soberano absorbe el ruido circundante y se fusiona con otros sectores.

En este cuaderno estudiaremos en profundidad los dos algoritmos que resuelven el problema de la densidad multiescala: **OPTICS** (*Ordering Points To Identify the Clustering Structure*, Ankerst et al., 1999) y la formulación jerárquica completa de **HDBSCAN** (Campello et al., 2013).

---

### Contenidos
1. La Patología de la Densidad Multiescala en Portafolios Multi-Activo
2. Fundamentos Matemáticos de **OPTICS**: Distancia de Núcleo (*Core Distance*) y Distancia de Alcanzabilidad (*Reachability Distance*)
3. Construcción e Interpretación del **Gráfico de Alcanzabilidad (*Reachability Plot*)**
4. Extracción de Clusters Multiescala con el Método $\xi$ (`cluster_method='xi'`) vs. Corte Fijo (`dbscan`)
5. Arquitectura Interna de **HDBSCAN**: Distancia de Alcance Mutuo, Árbol de Expansión Mínima (MST) y Persistencia $\lambda$
6. Agrupamiento Difuso (*Soft Clustering*) y Detección de Outliers Locales con **GLOSH**
7. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import DBSCAN, OPTICS, HDBSCAN
from sklearn.metrics import adjusted_rand_score, silhouette_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Simulación Financiera: Portafolio Multi-Activo con Densidades Variables

Construyamos un universo financiero de **480 activos** distribuidos en **3 regímenes con densidades espaciales muy diferentes** más 25 eventos extremos de mercado:
1. **Régimen 0 (Tesorería Soberana G7 - Ultra Denso)**: Dispersion mínima ($\sigma \approx 0.18$).
2. **Régimen 1 (Crédito Corporativo Industrial - Densidad Media)**: Dispersión moderada ($\sigma \approx 0.55$).
3. **Régimen 2 (Venture Capital / Biotech & Crypto - Baja Densidad)**: Alta dispersión natural ($\sigma \approx 1.35$).


In [ ]:
def generar_mercado_multiescala(seed=42):
    rng = np.random.default_rng(seed)
    # Regimen 0: Ultra-compacto (n=160)
    r0 = rng.normal(loc=[1.0, 4.5, 12.0, 2.0], scale=[0.15, 0.20, 0.8, 0.2], size=(160, 4))
    # Regimen 1: Densidad intermedia (n=160)
    r1 = rng.normal(loc=[3.2, 6.8, 24.0, 6.5], scale=[0.45, 0.55, 2.5, 0.8], size=(160, 4))
    # Regimen 2: Muy disperso pero coherente (n=135)
    r2 = rng.normal(loc=[7.5, 14.0, 58.0, 18.0], scale=[1.30, 1.80, 7.5, 2.8], size=(135, 4))
    # Ruido / Cisnes Negros (n=25)
    ruido = rng.uniform(low=[-2.0, 0.0, 5.0, 0.5], high=[13.0, 25.0, 95.0, 35.0], size=(25, 4))
    
    X = np.vstack([r0, r1, r2, ruido])
    y = np.array([0]*160 + [1]*160 + [2]*135 + [-1]*25)
    cols = ['beta_mercado', 'yield_efectivo_pct', 'volatilidad_anual_pct', 'liquidez_amihud_bps']
    return pd.DataFrame(X, columns=cols), y

df_multi, y_multi_real = generar_mercado_multiescala()
X_multi_rob = RobustScaler().fit_transform(df_multi)

# Veamos como DBSCAN con eps pequeno vs eps grande falla en capturar los 3 regimenes a la vez
db_peq = DBSCAN(eps=0.35, min_samples=10).fit(X_multi_rob)
db_gra = DBSCAN(eps=0.95, min_samples=10).fit(X_multi_rob)

print(f"DBSCAN (eps=0.35 - Estricto): Clusters = {len(set(db_peq.labels_))-1} | Ruido = {(db_peq.labels_==-1).sum()} | ARI = {adjusted_rand_score(y_multi_real, db_peq.labels_):.4f}")
print(f"DBSCAN (eps=0.95 - Laxo):     Clusters = {len(set(db_gra.labels_))-1} | Ruido = {(db_gra.labels_==-1).sum()} | ARI = {adjusted_rand_score(y_multi_real, db_gra.labels_):.4f}")


## 2. Fundamentos Matemáticos de OPTICS

**OPTICS** no produce una partición única rígida, sino que **ordena linealmente todas las observaciones del dataset** de acuerdo con su estructura espacial de densidad, almacenando dos cantidades para cada punto $p$:

### 1. Distancia de Núcleo (*Core Distance*, $d_{core}(p)$)
Es el radio mínimo $\varepsilon'$ necesario para que el punto $p$ reúna exactamente $MinPts$ vecinos y se convierta en Punto Núcleo:
$$d_{core}(p) = \begin{cases} \text{Indefinido } (\infty) & \text{si } ert{}N_{\varepsilon_{max}}(p)ert{} < MinPts \\ \text{Distancia al } MinPts\text{-ésimo vecino más cercano} & \text{en otro caso} \end{cases}$$

### 2. Distancia de Alcanzabilidad (*Reachability Distance*, $d_{reach}(q \leftarrow p)$)
La distancia de alcanzabilidad de un punto $q$ respecto a un Punto Núcleo $p$ es el máximo entre la distancia de núcleo de $p$ y la distancia métrica real entre $p$ y $q$:
$$d_{reach}(q \leftarrow p) = \max\Big\{ d_{core}(p), \; \text{dist}(p, q) \Big\}$$

Al recorrer el espacio priorizando siempre las menores distancias de alcanzabilidad mediante una cola de prioridad, OPTICS construye el **Gráfico de Alcanzabilidad (*Reachability Plot*)**:
- **Cada valle profundo en el gráfico corresponde a un cluster**: mientras más profundo es el valle, más denso es el régimen.
- **Los picos verticales entre valles** representan las fronteras vacías y los *outliers*.


In [ ]:
# Ajuste de OPTICS con extraccion adaptativa Xi (detecta caidas relativas de pendiente en los valles)
optics_model = OPTICS(min_samples=12, xi=0.06, min_cluster_size=0.08)
optics_model.fit(X_multi_rob)

# Extraccion de la secuencia ordenada y las distancias de alcanzabilidad
orden = optics_model.ordering_
reachability = optics_model.reachability_[orden]
etiquetas_ordenadas = optics_model.labels_[orden]

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Panel 1: Reachability Plot (Valles de Densidad)
x_idx = np.arange(len(X_multi_rob))
colores = ['#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
for klass, color in zip(range(3), colores):
    mask = (etiquetas_ordenadas == klass)
    axes[0].bar(x_idx[mask], reachability[mask], width=1.0, color=color, alpha=0.85, label=f'Valle Regimen {klass}')

mask_ruido = (etiquetas_ordenadas == -1)
axes[0].plot(x_idx[mask_ruido], reachability[mask_ruido], 'k.', markersize=5, label='Picos / Transiciones / Ruido (-1)')
axes[0].set_title('Gráfico de Alcanzabilidad de OPTICS (3 Valles a Distintas Alturas)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Orden Topológico de Procesamiento de Activos', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Distancia de Alcanzabilidad (Reachability Distance)', fontsize=10, fontweight='bold')
axes[0].legend(loc='upper left')

# Panel 2: Clusters extraidos por el metodo Xi en el espacio financiero
for klass, color in zip(range(3), colores):
    m = (optics_model.labels_ == klass)
    axes[1].scatter(df_multi.loc[m, 'beta_mercado'], df_multi.loc[m, 'volatilidad_anual_pct'],
                    c=color, s=45, edgecolor='k', alpha=0.85, label=f'Regimen {klass} (n={m.sum()})')
m_out = (optics_model.labels_ == -1)
axes[1].scatter(df_multi.loc[m_out, 'beta_mercado'], df_multi.loc[m_out, 'volatilidad_anual_pct'],
                c='#d62728', marker='x', s=65, lw=2, label=f'Ruido / Transicion (n={m_out.sum()})')
axes[1].set_title(f'OPTICS (Metodo Xi): Captura los 3 Regímenes de Distinta Densidad\n(ARI = {adjusted_rand_score(y_multi_real, optics_model.labels_):.3f})',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Beta de Mercado', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Volatilidad Anual (%)', fontsize=10, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()


## 3. Arquitectura Matemática de HDBSCAN y Soft Clustering

**HDBSCAN** lleva el concepto de densidad variable un paso más allá:
1. Define la **Distancia de Alcance Mutuo (*Mutual Reachability Distance*)** simétrica entre cualesquiera dos puntos $a$ y $b$:
   $$d_{mreach}(a, b) = \max\Big\{ d_{core}(a), \; d_{core}(b), \; \text{dist}(a, b) \Big\}$$
   Esto deja intacta la distancia entre puntos densos, pero "empuja" hacia lejos a los puntos aislados situados en zonas de baja densidad.
2. Construye el **Árbol de Expansión Mínima (*Minimum Spanning Tree*, MST)** mediante el algoritmo de Prim sobre el grafo completo ponderado por $d_{mreach}$.
3. Define $\lambda = \frac{1}{\varepsilon}$ y calcula la **Estabilidad / Persistencia de cada Cluster** a lo largo de su vida desde que nace ($\lambda_{birth}$) hasta que se divide o muere ($\lambda_{death}(p)$):
   $$S(C_k) = \sum_{p \in C_k} \big( \lambda_{death}(p) - \lambda_{birth}(C_k) \big)$$
4. Selecciona automáticamente los clusters que maximizan la suma de estabilidades en el árbol condensado.


In [ ]:
# Comparativa Final sobre el Portafolio Multiescala: DBSCAN vs OPTICS vs HDBSCAN
hdb_multi = HDBSCAN(min_cluster_size=25, min_samples=10, store_centers='medoid')
labels_hdb = hdb_multi.fit_predict(X_multi_rob)

print("=" * 78)
print("COMPARATIVA EN MERCADOS CON DENSIDAD MULTIESCALA (ARI vs Verdad Fundamental)")
print("=" * 78)
print(f"  1. DBSCAN (eps=0.35 fijo):  ARI = {adjusted_rand_score(y_multi_real, db_peq.labels_):.4f} (Rompe el cluster disperso)")
print(f"  2. DBSCAN (eps=0.95 fijo):  ARI = {adjusted_rand_score(y_multi_real, db_gra.labels_):.4f} (Fusiona clusters densos)")
print(f"  3. OPTICS (Metodo Xi):      ARI = {adjusted_rand_score(y_multi_real, optics_model.labels_):.4f}")
print(f"  4. HDBSCAN (Estabilidad):   ARI = {adjusted_rand_score(y_multi_real, labels_hdb):.4f} (Recuperacion casi perfecta)")
print("=" * 78)

# Inspeccion de los Medoides representativos extraidos nativamente por HDBSCAN
df_medoides_hdb = pd.DataFrame(
    RobustScaler().fit(df_multi).inverse_transform(hdb_multi.medoids_),
    columns=df_multi.columns,
    index=[f'Medoide Regimen {i}' for i in range(len(hdb_multi.medoids_))]
)
print("\nMedoides Financieros Nativos Calculados por HDBSCAN (store_centers='medoid'):")
print(df_medoides_hdb.round(2))


## 4. Conclusiones

1. **Gráfico de Alcanzabilidad (`OPTICS`)**: Es la radiografía topológica definitiva de un dataset. Si observas valles con profundidades muy distintas, significa que un único $\varepsilon$ de DBSCAN jamás podrá capturar todos los grupos simultáneamente.
2. **`HDBSCAN` con `store_centers='medoid'`**: En Scikit-Learn moderno, HDBSCAN no solo resuelve el problema de densidad variable maximizando la persistencia $\lambda$, sino que además permite calcular directamente los **medoides** de cada régimen y las probabilidades de pertenencia (`probabilities_`).
